# B-spline KAN on SRSD Feynman Medium
**26CSAI10H · Artificial Neural Networks · Group 13 · Phase 2**
**Abdelrahman Elbehery (246312)** — B-spline KAN, S2 training harness (`src/train.py`), S5 symbolic tool (`src/symbolic.py`)

| Section | What it does | Output (on Drive) |
|---|---|---|
| 1. Setup | Drive, code, packages | — |
| 2. Data | Downloads SRSD Feynman Medium once | `data/srsd-feynman_medium/` |
| 3. Checks | Test suite + versions for Section V | — |
| 4. Configuration | Specs from `configs/bkan/*.yaml`, training budget | — |
| 5. Timing pilot | Seconds per epoch on GPU / CPU | numbers for the budget |
| 6. Accuracy runs | KAN-1, KAN-4, KAN-2 (KAN-3 optional), 3 seeds, shared Adam harness | `results/bkan.csv`, `results/predictions/` |
| 7. Symbolic pipeline | LBFGS + sparsity → prune → snap → refit → score | `results/bkan_recovery.csv` |
| 8. Edge-function figure | Pruned, symbolic KAN of one recovered law | `figures/bkan_edges_<eq>.pdf/.png` |
| 9. Results | Spec table, grid-extension staircase, product laws, recovery | `figures/*.pdf` |
| 10. Export | Zips results and figures | `bkan_results.zip` |

**Evaluation protocol.** Training uses standardised targets (SRSD Medium targets span about 1e-37 to 2e38,
which underflow or overflow in float32). Every metric is computed **after** undoing that scaling:
test split, original units, float64. With $e_i=\hat y_i-y_i$ over the $n$ test points:

| Metric | Definition | Units |
|---|---|---|
| MSE | $\frac1n\sum_i e_i^2$ | target units² |
| RMSE | $\sqrt{\text{MSE}}$ | target units |
| MAE | $\frac1n\sum_i \lvert e_i\rvert$ | target units |
| R² | $1-\text{MSE}/\operatorname{Var}(y_\text{test})$ | none |
| NRMSE | $\text{RMSE}/\operatorname{std}(y_\text{test})$ (Petersen et al., 2021) $=\sqrt{1-R^2}$ | none |

MSE, RMSE and MAE are reported per equation (their units differ between equations); NRMSE (headline) and R²
are also summarised across equations.

**Resuming.** Every loop appends one row per finished run to a CSV on Drive and skips rows already there.
After a disconnect: run Section 1 and Section 4, then the section you were in.

## 1. Setup
**Before the first run:** Runtime → Change runtime type → **T4 GPU**. The code is cloned from GitHub to
`/content` (fast, temporary); data and results live on Drive in `MyDrive/ann-group13/` (they survive disconnects).
If the code is in a folder on Drive (with `src/`, `notebooks/`, ... inside `MyDrive/ann-group13/`), set `CODE_SOURCE = "drive"`; for `bkan_elbehery.zip` on Drive, `"zip"`.

In [1]:
import os, sys, subprocess, zipfile

IN_COLAB = "google.colab" in sys.modules
SMOKE = os.environ.get("BKAN_SMOKE") == "1"          # tiny end-to-end check; leave off for real runs

CODE_SOURCE = "github"                                # "github" (team repo), "drive" (folder on Drive) or "zip"
REPO_URL = "https://github.com/Elbehery-237/ann-group13.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/ann-group13"       # data/, results/, figures/ live here
    os.makedirs(WORK, exist_ok=True)
    if CODE_SOURCE == "drive":                        # code already in a folder on Drive
        candidates = [WORK, f"{WORK}/bkan_elbehery", f"{WORK}/ann-group13", "/content/drive/MyDrive/bkan_elbehery"]
        PROJECT = next((p for p in candidates if os.path.exists(f"{p}/src/train.py")), None)
        if PROJECT is None:
            raise FileNotFoundError("No folder with src/train.py found in: " + ", ".join(candidates)
                                    + ". Set PROJECT to your folder, or use CODE_SOURCE = 'github'.")
    elif CODE_SOURCE == "zip":
        PROJECT = "/content/bkan_elbehery"
        ZIP = f"{WORK}/bkan_elbehery.zip"
        if not os.path.exists(PROJECT):
            if not os.path.exists(ZIP):
                raise FileNotFoundError(f"Upload bkan_elbehery.zip to {WORK}/ first")
            zipfile.ZipFile(ZIP).extractall("/content")
    else:
        PROJECT = "/content/ann-group13"
        if os.path.exists(PROJECT):
            subprocess.run(["git", "-C", PROJECT, "pull", "-q"], check=True)
        else:
            subprocess.run(["git", "clone", "-q", REPO_URL, PROJECT], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{PROJECT}/requirements.txt"], check=True)
else:                                                 # local run (e.g. a laptop with a GPU)
    PROJECT = os.environ.get("BKAN_PROJECT", os.path.abspath(".."))
    WORK = os.environ.get("BKAN_WORK", PROJECT)

os.chdir(PROJECT)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
os.environ.setdefault("SRSD_ROOT", f"{WORK}/data/srsd-feynman_medium")
SRSD_ROOT = os.environ["SRSD_ROOT"]
RESULTS, FIGURES = f"{WORK}/results", f"{WORK}/figures"
PREDS = f"{RESULTS}/predictions"
for p in (RESULTS, FIGURES, PREDS):
    os.makedirs(p, exist_ok=True)

import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("project:", PROJECT, "| work:", WORK, "| device:", DEVICE, "| smoke:", SMOKE)

Mounted at /content/drive
project: /content/ann-group13 | work: /content/drive/MyDrive/ann-group13 | device: cuda | smoke: False


## 2. Data: SRSD Feynman Medium (downloaded once)
40 problems, 1–6 inputs, 8000 / 1000 / 1000 rows (train / val / test), ground truth in `true_eq/*.pkl`.

In [2]:
if not os.path.exists(f"{SRSD_ROOT}/true_eq"):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="yoshitomo-matsubara/srsd-feynman_medium", repo_type="dataset",
                      local_dir=SRSD_ROOT)
from src.bkan_data import list_problems
print(len(list_problems()), "equations in", SRSD_ROOT)          # expected: 40

40 equations in /content/drive/MyDrive/ann-group13/data/srsd-feynman_medium


## 3. Checks: tests and versions
All tests should pass before training (about 3 minutes; data-dependent tests run on the real files).
Copy the printed versions into Section V of the paper.

In [3]:
RUN_TESTS = not SMOKE
if RUN_TESTS:
    r = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])

........................................................................ [ 50%]
..............................F........F..............................   [100%]
=================================== FAILURES ===================================
_________________ test_to_original_round_trip[feynman-i.39.10] _________________

name = 'feynman-i.39.10'

    @needs_data
    @pytest.mark.parametrize("name", PROBLEMS)
    def test_to_original_round_trip(name):
        data = load_srsd(name)
        true = load_true_eq(name, SRSD_ROOT)
        d = data["d"]
        back = to_original(scaled_form(true, data["xscaler"], data["yscaler"], d),
                           data["xscaler"], data["yscaler"], d)
        test = data["raw"]["test"]
        np.testing.assert_allclose(predict(back, test[:, :d]), test[:, d], rtol=1e-6,
                                   atol=1e-9 * np.abs(test[:, d]).max())
>       assert is_exact(back, true, round_digits=6)
E       assert False
E        +  where False = is_exac

In [4]:
import platform, numpy, sympy, kan
from importlib.metadata import version
print("python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("numpy", numpy.__version__, "| sympy", sympy.__version__, "| pykan", version("pykan"), "| zss", version("zss"))

python 3.13.16 | torch 2.14.1+cu130 | CUDA 13.0 | GPU NVIDIA A100-SXM4-80GB
numpy 2.4.6 | sympy 1.14.0 | pykan 0.2.8 | zss 1.2.0


## 4. Configuration
Specs come from `configs/bkan/*.yaml`. The training budget (`max_epochs`, `patience`) is the team-wide value
in `src/train.py → DEFAULTS`, the same for all four models; set it from the timing pilot (Section 5).

In [5]:
import time, glob, yaml
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt

from src.train import DEFAULTS, set_seed, train                                   # S2 (mine)
from src.symbolic import load_true_eq, to_original, is_exact, ned, predict        # S5 (mine)
from src.models.bspline_kan import BSplineKAN, grid_update_hook
from src.bkan_data import load_problem, list_problems
from src.bkan_eval import (regression_metrics, test_metrics, predict_original, save_predictions,
                           per_equation, summarize)

def cfg(name):
    with open(f"configs/bkan/{name}.yaml") as f:
        return yaml.safe_load(f)

def hp(c):                       # yaml -> BSplineKAN keyword arguments
    return dict(hidden=tuple(c["hidden"]), grid=c["grid"], k=c["k"])

KAN1, KAN2, KAN3, KAN4, SYM = (cfg(n) for n in ("kan1", "kan2", "kan3", "kan4", "symbolic"))
RUN_KAN3 = False                 # optional: only after KAN-1, KAN-2 and KAN-4 are complete

SEEDS = (0, 1, 2)
EQUATIONS = list_problems()      # all 40 (or a fixed subset chosen BEFORE seeing results; list it in the paper)
BUDGET = dict(max_epochs=DEFAULTS["max_epochs"], patience=DEFAULTS["patience"], lr=DEFAULTS["lr"])

# Product laws named in the KAN-4 hypothesis
PRODUCT_LAWS = ["feynman-i.39.10", "feynman-i.34.8", "feynman-ii.34.2", "feynman-i.43.31", "feynman-i.12.2"]

if SMOKE:                        # tiny settings for an end-to-end code check only
    EQUATIONS, SEEDS = EQUATIONS[:2], (0,)
    BUDGET.update(max_epochs=30, patience=10)
    KAN2["grids"] = [5, 10]
    SYM.update(lambdas=[1e-3], steps_sparse=20, steps_after_prune=10, steps_refit=5)

print(len(EQUATIONS), "equations | seeds", SEEDS, "| budget", BUDGET)

40 equations | seeds (0, 1, 2) | budget {'max_epochs': 5000, 'patience': 300, 'lr': 0.001}


In [6]:
import contextlib, io

@contextlib.contextmanager
def quiet():
    # hide pykan's progress bars and 'saving model version' messages
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        yield

def append_row(path, row):
    pd.DataFrame([row]).to_csv(path, mode="a", header=not os.path.exists(path), index=False)

def finished(path, keys=("spec", "equation", "seed")):
    if not os.path.exists(path):
        return set()
    return set(pd.read_csv(path)[list(keys)].itertuples(index=False, name=None))

## 5. Timing pilot (plan step 6)
KAN-1 for 200 epochs on three equations with 1, 3 and 6 inputs, on GPU and on CPU (small models are
sometimes as fast on CPU). Use the result to agree `max_epochs` and `patience` with the team.

In [7]:
RUN_PILOT = not SMOKE
PILOT = ["feynman-i.29.4", "feynman-i.12.2", "feynman-i.11.19"]          # d = 1, 3, 6
if RUN_PILOT:
    rows = []
    for dev in (["cuda", "cpu"] if torch.cuda.is_available() else ["cpu"]):
        for eq in PILOT:
            set_seed(0); data = load_problem(eq)
            m = BSplineKAN(data["d"], **hp(KAN1), seed=0, device=dev)
            _, _, info = train(m, data, max_epochs=200, patience=10**9, hooks=(grid_update_hook,), device=dev)
            rows.append(dict(device=dev, equation=eq, d=data["d"], s_per_epoch=info["time"] / info["epochs"]))
    pilot = pd.DataFrame(rows); display(pilot)

    # runs = 40 eq x 3 seeds x (KAN-1 + KAN-4 + 3 KAN-2 stages); early stopping makes this an upper bound
    s_ep = pilot.groupby("device").s_per_epoch.mean().min()
    for ep in (500, 1000, 2000, 5000):
        print(f"max_epochs={ep:>5}: at most {40 * 3 * 5 * ep * s_ep / 3600:6.1f} h")

,device,equation,d,s_per_epoch
0,cuda,feynman-i.29.4,1,0.064242
1,cuda,feynman-i.12.2,3,0.039601
2,cuda,feynman-i.11.19,6,0.055842
3,cpu,feynman-i.29.4,1,0.033386
4,cpu,feynman-i.12.2,3,0.041246
5,cpu,feynman-i.11.19,6,0.056704


max_epochs=  500: at most    3.6 h
max_epochs= 1000: at most    7.3 h
max_epochs= 2000: at most   14.6 h
max_epochs= 5000: at most   36.5 h


## 6. Accuracy runs (plan step 7)
All accuracy numbers come from the shared Adam harness `train()` (same optimizer, loss, budget and early
stopping as the other three models). Order: KAN-1 → KAN-4 → KAN-2 (→ KAN-3 if time).
KAN-2 writes one row per grid stage (`KAN-2/G=5`, `KAN-2/G=10`, `KAN-2/G=20`) for the staircase curve.
Diverged runs stay in the file with `diverged=True`. Test predictions (original units) are saved so the
group's final notebook can recompute every metric with the same code for all models.

In [7]:
OUT = f"{RESULTS}/bkan.csv"

def record(spec, c, grid, eq, seed, model, data, info):
    metrics, y_pred = test_metrics(model, data, device=DEVICE, return_pred=True)
    save_predictions(PREDS, "bkan", spec, eq, seed, y_pred)
    append_row(OUT, dict(model="bkan", spec=spec, grid=grid, k=c["k"], hidden=str(c["hidden"]),
                         equation=eq, d=data["d"], seed=seed, n_params=model.count_params(),
                         train_time_s=info["time"], epochs_run=info["epochs"], best_epoch=info["best_epoch"],
                         diverged=info["diverged"], val_mse_scaled=info["best_val"], **metrics))
    print(f"{spec:<11} {eq:<20} seed={seed} epochs={info['epochs']:>5} "
          f"MSE={metrics['mse']:.3g} RMSE={metrics['rmse']:.3g} MAE={metrics['mae']:.3g} "
          f"R2={metrics['r2']:.4f} NRMSE={metrics['nrmse']:.3g} diverged={info['diverged']}")

def run_single(c):
    done = finished(OUT)
    for eq in EQUATIONS:
        for seed in SEEDS:
            if (c["spec"], eq, seed) in done:
                continue
            set_seed(seed); data = load_problem(eq)
            model = BSplineKAN(data["d"], **hp(c), seed=seed, device=DEVICE)
            model, hist, info = train(model, data, hooks=(grid_update_hook,), device=DEVICE, **BUDGET)
            record(c["spec"], c, c["grid"], eq, seed, model, data, info)

def run_grid_extension(c):
    # refine() returns a new model with new parameters, so train() (fresh Adam) runs once per stage
    grids = c["grids"]
    done = finished(OUT)
    for eq in EQUATIONS:
        for seed in SEEDS:
            if (f"KAN-2/G={grids[-1]}", eq, seed) in done:
                continue
            set_seed(seed); data = load_problem(eq)
            model = BSplineKAN(data["d"], **hp(c), seed=seed, device=DEVICE)
            for s, G in enumerate(grids):
                if s > 0:
                    model.kan = model.kan.refine(G)
                model, hist, info = train(model, data, hooks=(grid_update_hook,) if s == 0 else (),
                                          device=DEVICE, **BUDGET)
                if (f"KAN-2/G={G}", eq, seed) not in done:
                    record(f"KAN-2/G={G}", c, G, eq, seed, model, data, info)
                if info["diverged"]:
                    break

NameError: name 'RESULTS' is not defined

In [9]:
run_single(KAN1)

KAN-1       feynman-i.13.4       seed=2 epochs= 5000 MSE=0.563 RMSE=0.75 MAE=0.511 R2=0.9999 NRMSE=0.0113 diverged=False
KAN-1       feynman-i.15.10      seed=0 epochs= 5000 MSE=1.36e+08 RMSE=1.17e+04 MAE=7.9e+03 R2=0.9999 NRMSE=0.0114 diverged=False
KAN-1       feynman-i.15.10      seed=1 epochs= 5000 MSE=1.26e+08 RMSE=1.12e+04 MAE=7.02e+03 R2=0.9999 NRMSE=0.0109 diverged=False
KAN-1       feynman-i.15.10      seed=2 epochs= 5000 MSE=6.58e+07 RMSE=8.11e+03 MAE=5.33e+03 R2=0.9999 NRMSE=0.00789 diverged=False
KAN-1       feynman-i.16.6       seed=0 epochs= 5000 MSE=6.06e+10 RMSE=2.46e+05 MAE=1.49e+05 R2=1.0000 NRMSE=0.00535 diverged=False
KAN-1       feynman-i.16.6       seed=1 epochs= 5000 MSE=1.7e+10 RMSE=1.3e+05 MAE=6.56e+04 R2=1.0000 NRMSE=0.00283 diverged=False
KAN-1       feynman-i.16.6       seed=2 epochs= 5000 MSE=2.25e+10 RMSE=1.5e+05 MAE=8.18e+04 R2=1.0000 NRMSE=0.00325 diverged=False
KAN-1       feynman-i.18.4       seed=0 epochs= 5000 MSE=0.0106 RMSE=0.103 MAE=0.0604 R2=0.99

In [11]:
run_single(KAN4)

KAN-4       feynman-i.34.10      seed=1 epochs= 5000 MSE=8.5e+15 RMSE=9.22e+07 MAE=5.16e+07 R2=1.0000 NRMSE=0.00349 diverged=False
KAN-4       feynman-i.34.10      seed=2 epochs= 5000 MSE=2.17e+15 RMSE=4.66e+07 MAE=2.01e+07 R2=1.0000 NRMSE=0.00176 diverged=False
KAN-4       feynman-i.34.27      seed=0 epochs= 5000 MSE=2.17e-53 RMSE=4.66e-27 MAE=2.41e-27 R2=1.0000 NRMSE=0.00178 diverged=False
KAN-4       feynman-i.34.27      seed=1 epochs= 5000 MSE=6.38e-53 RMSE=7.99e-27 MAE=4.57e-27 R2=1.0000 NRMSE=0.00305 diverged=False
KAN-4       feynman-i.34.27      seed=2 epochs= 5000 MSE=6.62e-53 RMSE=8.14e-27 MAE=4.78e-27 R2=1.0000 NRMSE=0.00311 diverged=False
KAN-4       feynman-i.34.8       seed=0 epochs= 1097 MSE=2.75e-21 RMSE=5.25e-11 MAE=2.74e-11 R2=0.5351 NRMSE=0.682 diverged=False
KAN-4       feynman-i.34.8       seed=1 epochs=  810 MSE=4.34e-21 RMSE=6.59e-11 MAE=2.99e-11 R2=0.2663 NRMSE=0.857 diverged=False
KAN-4       feynman-i.34.8       seed=2 epochs= 2203 MSE=5.27e-21 RMSE=7.26e-11 M

In [3]:
run_grid_extension(KAN2)

NameError: name 'run_grid_extension' is not defined

In [ ]:
if RUN_KAN3:
    run_single(KAN3)

## 7. Symbolic pipeline (plan steps 8–9)
A separate run from the accuracy runs (state this in the paper): pykan's LBFGS `fit()` with a sparsity penalty,
λ chosen on **validation** NRMSE, then prune, retrain briefly, snap every edge to the closest library function,
refit the affine constants, convert the formula back to the original variables and score it against the
ground truth (`is_exact`, SRBench convention; `ned`, SRSD procedure). Small models, so a **CPU runtime** in a
second Colab tab can run this while Section 6 runs on the GPU. Pilot the three equations first.

In [ ]:
OUT_SYM = f"{RESULTS}/bkan_recovery.csv"
NOLOG = 10**9                                       # pykan 0.2.8: fit(log=0) raises ZeroDivisionError
PILOT_SYM = ["feynman-i.39.10", "feynman-i.11.19", "feynman-iii.17.37"]   # product, sum of products, cos
EQUATIONS_SYM = [e for e in PILOT_SYM if e in EQUATIONS] if not SMOKE else EQUATIONS
# EQUATIONS_SYM = EQUATIONS                         # after checking the pilot formulas look sensible

def kan_dataset(data):                              # pykan calls our validation split "test"
    return dict(train_input=data["Xtr"], train_label=data["ytr"],
                test_input=data["Xva"], test_label=data["yva"])

def nrmse_orig(m, data, split):                     # same protocol as the accuracy runs
    y_pred, _ = predict_original(m, data[f"X{split}"], data["yscaler"])
    name = {"va": "val", "te": "test"}[split]
    return regression_metrics(data["raw"][name][:, -1], y_pred)["nrmse"]

def fit_sparse(data, lamb):
    set_seed(0)
    m = BSplineKAN(data["d"], **hp(SYM), seed=0)
    with quiet():
        m.kan.fit(kan_dataset(data), opt="LBFGS", steps=SYM["steps_sparse"], lamb=lamb, log=NOLOG)
    return m

def symbolic_run(eq):
    data = load_problem(eq)
    ds = kan_dataset(data)
    best = None
    for lamb in SYM["lambdas"]:
        m = fit_sparse(data, lamb)
        va = nrmse_orig(m, data, "va")
        if best is None or va < best[0]:
            best = (va, lamb, m)
    _, lamb, m = best
    row = dict(equation=eq, d=data["d"], lamb=lamb, test_nrmse_unpruned=nrmse_orig(m, data, "te"))
    with quiet():
        m.kan = m.kan.prune()
        m.kan.fit(ds, opt="LBFGS", steps=SYM["steps_after_prune"], log=NOLOG)
    row["test_nrmse_pruned"] = nrmse_orig(m, data, "te")
    row["n_params_pruned"] = m.count_params()
    with quiet():
        m.kan.auto_symbolic(lib=SYM["library"], verbose=0)
        m.kan.fit(ds, opt="LBFGS", steps=SYM["steps_refit"], update_grid=False, log=NOLOG)
    expr = to_original(m.formula(), data["xscaler"], data["yscaler"], data["d"])
    true = load_true_eq(eq, SRSD_ROOT)
    with np.errstate(all="ignore"):
        y_sym = predict(expr, data["raw"]["test"][:, :-1])
    row.update(recovered_formula=str(expr), true_formula=str(true),
               exact=is_exact(expr, true, timeout=SYM["simplify_timeout_s"], round_digits=SYM["round_digits"]),
               ned=ned(expr, true),
               **{f"symbolic_{k}": v for k, v in regression_metrics(data["raw"]["test"][:, -1], y_sym).items()
                  if k != "finite_pred"})
    return row

done_sym = finished(OUT_SYM, keys=("equation",))
for eq in EQUATIONS_SYM:
    if (eq,) in done_sym:
        continue
    t0 = time.perf_counter()
    try:
        row = symbolic_run(eq)
        row["error"] = ""
    except Exception as e:                           # record the failure, keep going
        row = dict(equation=eq, exact=False, ned=1.0, error=repr(e)[:300])
    row["time_s"] = time.perf_counter() - t0
    append_row(OUT_SYM, row)
    print(f"{eq:<20} exact={row['exact']} NED={row['ned']:.3f} {row.get('recovered_formula', row['error'])[:90]}")

## 8. Edge-function figure (plan step 10)
The pruned KAN of one **recovered** equation (a product law if possible), each edge drawn after snapping to
its symbolic function. Re-runs the pipeline for that one equation with the λ chosen in Section 7.

In [ ]:
rec = pd.read_csv(OUT_SYM)
exact = rec[rec.exact == True].equation.tolist()
FIG_EQ = next((e for e in PRODUCT_LAWS if e in exact), exact[0] if exact else rec.equation.iloc[0])
print("figure equation:", FIG_EQ, "| exact:", FIG_EQ in exact)

data = load_problem(FIG_EQ); ds = kan_dataset(data)
m = fit_sparse(data, float(rec.set_index("equation").loc[FIG_EQ, "lamb"]))
with quiet():
    m.kan = m.kan.prune(); m.kan.fit(ds, opt="LBFGS", steps=SYM["steps_after_prune"], log=NOLOG)
    m.kan.auto_symbolic(lib=SYM["library"], verbose=0)
m.kan(data["Xtr"])                                  # store activations for plotting
m.kan.plot(in_vars=[f"x{i}" for i in range(data["d"])], out_vars=["y"], folder="/tmp/kan_plot",
           title=f"{FIG_EQ} (pruned, symbolic)")
plt.savefig(f"{FIGURES}/bkan_edges_{FIG_EQ}.pdf", bbox_inches="tight")
plt.savefig(f"{FIGURES}/bkan_edges_{FIG_EQ}.png", bbox_inches="tight", dpi=200)
plt.show()
print("snapped formula (original variables):",
      to_original(m.formula(), data["xscaler"], data["yscaler"], data["d"]))

## 9. Results for Sections VI-B and VII-A
All five metrics per equation (median over the 3 seeds), then the cross-equation summary of the unit-free
NRMSE and R² (median and IQR). MSE / RMSE / MAE are not aggregated across equations: their units differ.

In [ ]:
res = pd.read_csv(OUT)
table = summarize(res)
display(table.round(4))
table.to_csv(f"{RESULTS}/bkan_summary.csv", index=False)

In [ ]:
# KAN-2 grid-extension staircase: median (IQR band) test NRMSE across equations vs grid size
k2 = res[res.spec.str.startswith("KAN-2")]
if len(k2):
    per_eq = k2.groupby(["grid", "equation"]).nrmse.median().reset_index()
    s = per_eq.groupby("grid").nrmse
    G, med, q1, q3 = s.median().index, s.median().values, s.quantile(.25).values, s.quantile(.75).values
    plt.figure(figsize=(4.5, 3.2))
    plt.plot(G, med, "o-", label="median"); plt.fill_between(G, q1, q3, alpha=.25, label="IQR")
    plt.xscale("log", base=2); plt.yscale("log"); plt.xticks(G, G)
    plt.xlabel("grid intervals G"); plt.ylabel("test NRMSE"); plt.title("KAN-2 grid extension")
    plt.legend(); plt.tight_layout(); plt.savefig(f"{FIGURES}/bkan_kan2_staircase.pdf"); plt.show()

In [ ]:
# KAN-1 vs KAN-4 (multiplication nodes) on the product laws
pl = res[res.equation.isin(PRODUCT_LAWS) & res.spec.isin(["KAN-1", "KAN-4"])]
if len(pl):
    with pd.option_context("display.float_format", "{:.4g}".format):
        display(per_equation(pl).set_index(["equation", "spec"])[["mse", "rmse", "mae", "r2", "nrmse"]])

In [ ]:
# Per-equation table: all five metrics (median over seeds), one row per (spec, equation)
pe = per_equation(res)
pe.to_csv(f"{RESULTS}/bkan_per_equation.csv", index=False)
with pd.option_context("display.float_format", "{:.4g}".format, "display.max_rows", 500):
    display(pe[["spec", "equation", "mse", "rmse", "mae", "r2", "nrmse", "n_params"]])

In [ ]:
rec = pd.read_csv(OUT_SYM)
print(f"exact recoveries: {int(rec.exact.sum())} / {len(rec)} | median NED: {rec.ned.median():.3f} "
      f"| round_digits = {SYM['round_digits']} (report in the paper)")
cols = [c for c in ["equation", "true_formula", "recovered_formula", "exact", "ned",
                    "test_nrmse_unpruned", "test_nrmse_pruned", "symbolic_mse", "symbolic_rmse", "symbolic_mae",
                    "symbolic_r2", "symbolic_nrmse", "error"] if c in rec]
display(rec[cols])

## 10. Export
Zips everything this notebook produced (CSVs, predictions, figures) into `bkan_results.zip` on Drive,
ready to hand to the group's final notebook.

In [ ]:
import shutil
stage = "/tmp/bkan_results"
shutil.rmtree(stage, ignore_errors=True)
shutil.copytree(RESULTS, f"{stage}/results", ignore=shutil.ignore_patterns("predictions"))
shutil.copytree(PREDS, f"{stage}/results/predictions")
shutil.copytree(FIGURES, f"{stage}/figures")
print(shutil.make_archive(f"{WORK}/bkan_results", "zip", stage))